In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("merging.csv"); df.head()

,Unnamed: 0,text,label
0,0,I think the issue of elderly drivers is a comp...,1
1,1,I agree that television violence has a negativ...,1
2,2,I understand the concerns of people who are op...,1
3,3,"Dear Principal, I am writing to you today to e...",1
4,4,"Yes, there is a cause that I actively support:...",1


In [3]:
x = df['text']
y = df['label']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 1000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding='post')

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense, Dropout, Bidirectional, LayerNormalization

model = Sequential([
    Embedding(vocab_size, 256, input_shape=(max_len, )),
    Bidirectional(LSTM(128)),
    Dropout(0.1),
    LayerNormalization(),
    Dense(128, activation='relu'),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 256)       │       256,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, 256)            │           512 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 687,809 (2.62 MB)

 Trainable params: 687,809 (2.62 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(
    X_train_pad, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)


Epoch 1/15


60/60 ━━━━━━━━━━━━━━━━━━━━ 14s 150ms/step - accuracy: 0.8939 - loss: 0.2598 - val_accuracy: 0.9641 - val_loss: 0.1295
Epoch 2/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 120ms/step - accuracy: 0.9493 - loss: 0.1394 - val_accuracy: 0.9641 - val_loss: 0.1382
Epoch 3/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 117ms/step - accuracy: 0.9768 - loss: 0.0607 - val_accuracy: 0.9768 - val_loss: 0.0885
Epoch 4/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 118ms/step - accuracy: 0.9446 - loss: 0.1625 - val_accuracy: 0.5781 - val_loss: 0.5752
Epoch 5/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 118ms/step - accuracy: 0.7392 - loss: 0.4881 - val_accuracy: 0.8376 - val_loss: 0.3168
Epoch 6/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 117ms/step - accuracy: 0.9303 - loss: 0.1946 - val_accuracy: 0.9198 - val_loss: 0.2852
Epoch 7/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 118ms/step - accuracy: 0.9604 - loss: 0.1084 - val_accuracy: 0.9768 - val_loss: 0.0771
Epoch 8/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 118ms/step - accuracy: 0.9873 - loss: 0.0456 - val_accuracy: 0.9768 - val

In [13]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=1)
print("Test Accuracy:", acc)


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - accuracy: 0.9848 - loss: 0.0665
Test Accuracy: 0.9848229289054871


In [ ]:
def predict_text(text):
    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')
    prob = model.predict(seq)[0][0]
    return prob, ("AI" if prob > 0.5 else "Human")

print(predict_text("After removing those models, what remains?You’ll mostly have:Text ML / DL classifiersTextBlob sentimentSpam detectionSimple vision (digits, emotion)Audio emotion (CPU-based)Recommendation (non-SVD)Lightweight CNN / classical MLThis becomes a light-to-medium ML Flask app"))
print(predict_text('Hi there iam Sufiyan and hows going on?'))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step
(np.float32(1.0), 'AI')
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
(np.float32(0.1277553), 'Human')


In [15]:
ai_inputs = input("Enter the ai input: ")
print(predict_text(ai_inputs))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
(np.float32(0.9999996), 'AI')


In [16]:
human_inputs = input("Enter the ai input: ")
print(predict_text(human_inputs))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
(np.float32(0.0045687757), 'Human')


In [17]:
model.save("deeplearning_version3_0.keras")


In [18]:
import pickle

with open("tokenizer_version3_0.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
